# 소희 알약 이미지 EDA

원본을 수정하지 않고 JSON 연결 → bbox 확인 → 무결성 → 분포 → 촬영 조건 → 분할 후보를 분석합니다.
실행 환경: 프로젝트의 uv 환경. 커널의 작업 폴더는 프로젝트 루트 또는 notebooks/를 지원합니다.
전체 JSON은 메타데이터만 읽고, 이미지는 한 장씩 읽어 메모리 사용을 제한합니다.
노트북 출력은 저장하지 않으며 요약 보고서와 PNG는 실행 시 생성됩니다. seed=42.
기존 data_preprocessing.ipynb 및 dongwook_eda.ipynb는 변경하지 않습니다.


In [30]:
import json
import os
import sys
from pathlib import Path

os.environ["WANDB_MODE"] = "disabled"
import pandas as pd
from IPython.display import Image, display

root = Path.cwd().resolve()
if not (root / "src" / "config.py").is_file():
    root = root.parent
if not (root / "src" / "config.py").is_file():
    raise RuntimeError("프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import PROJECT_ROOT  # noqa: E402  # 프로젝트 경로 설정 후 불러온다.

In [31]:
import hashlib
import math

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

CONDITIONS = ("drug_N", "drug_S", "back_color", "drug_dir", "light_color")
RECORD_COLUMNS = [
    "source",
    "image_id",
    "file_name",
    "category_id",
    "category_name",
    "width",
    "height",
    "x",
    "y",
    "bbox_width",
    "bbox_height",
    "bbox_valid",
    "area_matches",
    *CONDITIONS,
]


def image_paths(directory: Path) -> list[Path]:
    """지원하는 이미지 경로를 정렬해 반환한다."""
    if not directory.is_dir():
        raise FileNotFoundError(f"이미지 폴더가 없습니다: {directory}")
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    return sorted(path for path in directory.rglob("*") if path.suffix.lower() in extensions)


def as_objects(value: object) -> list[dict]:
    """JSON의 단일 객체와 객체 목록을 같은 형태로 다룬다."""
    if isinstance(value, dict):
        return [value]
    if isinstance(value, list) and all(isinstance(item, dict) for item in value):
        return value
    raise ValueError("객체 또는 객체 목록이 필요합니다.")


def load_annotations(directory: Path) -> tuple[pd.DataFrame, list[Path], pd.DataFrame]:
    """JSON 내부 참조로 객체를 연결하고 읽기 및 참조 오류를 기록한다."""
    if not directory.is_dir():
        raise FileNotFoundError(f"라벨 폴더가 없습니다: {directory}")
    files = sorted(directory.rglob("*.json"))
    rows, issues = [], []
    for path in files:
        source = (
            str(path.relative_to(PROJECT_ROOT)) if path.is_relative_to(PROJECT_ROOT) else str(path)
        )
        try:
            payload = json.loads(path.read_text(encoding="utf-8-sig"))
            images = as_objects(payload["images"])
            annotations = as_objects(payload["annotations"])
            categories = as_objects(payload["categories"])
            for annotation in annotations:
                try:
                    matches = [item for item in images if item.get("id") == annotation["image_id"]]
                    if len(matches) != 1:
                        raise ValueError("JSON 내부 image_id 참조가 없거나 중복됩니다.")
                    img = matches[0]
                    category_id = annotation["category_id"]
                    matches = [item for item in categories if item.get("id") == category_id]
                    if len(matches) != 1:
                        raise ValueError("JSON 내부 category_id 참조가 없거나 중복됩니다.")
                    width, height = float(img["width"]), float(img["height"])
                    bbox = annotation.get("bbox")
                    try:
                        x, y, bbox_width, bbox_height = map(float, bbox)
                    except (TypeError, ValueError):
                        x = y = bbox_width = bbox_height = float("nan")
                    valid = (
                        all(
                            math.isfinite(value)
                            for value in (width, height, x, y, bbox_width, bbox_height)
                        )
                        and width > 0
                        and height > 0
                        and x >= 0
                        and y >= 0
                        and bbox_width > 0
                        and bbox_height > 0
                        and x + bbox_width <= width
                        and y + bbox_height <= height
                    )
                    try:
                        area_matches = math.isclose(
                            float(annotation["area"]),
                            bbox_width * bbox_height,
                            rel_tol=1e-3,
                            abs_tol=1.0,
                        )
                    except (KeyError, TypeError, ValueError):
                        area_matches = False
                    rows.append(
                        {
                            "source": source,
                            "image_id": annotation["image_id"],
                            "file_name": Path(img["file_name"]).name,
                            "category_id": category_id,
                            "category_name": matches[0].get("name")
                            or img.get("dl_name")
                            or str(category_id),
                            "width": width,
                            "height": height,
                            "x": x,
                            "y": y,
                            "bbox_width": bbox_width,
                            "bbox_height": bbox_height,
                            "bbox_valid": valid,
                            "area_matches": area_matches,
                            **{key: img.get(key) for key in CONDITIONS},
                        }
                    )
                except (KeyError, TypeError, ValueError) as error:
                    issues.append({"source": source, "issue": str(error)})
        except (OSError, ValueError, KeyError, TypeError) as error:
            issues.append({"source": source, "issue": str(error)})
    return (
        pd.DataFrame(rows, columns=RECORD_COLUMNS),
        files,
        pd.DataFrame(issues, columns=["source", "issue"]),
    )


def prepare_tables(records: pd.DataFrame, paths: list[Path]) -> tuple[pd.DataFrame, dict]:
    """파일명이 유일한 이미지만 연결하고 중복 객체와 bbox 통계 열을 만든다."""
    candidates = {}
    for path in paths:
        candidates.setdefault(path.name, []).append(path)
    lookup = {name: values[0] for name, values in candidates.items() if len(values) == 1}
    result = records.copy()
    result["image_path"] = result.file_name.map(lookup)
    result["image_present"] = result.file_name.isin(lookup)
    result["ambiguous_image"] = result.file_name.map(lambda name: len(candidates.get(name, [])) > 1)
    result["duplicate"] = result.duplicated(
        ["file_name", "category_id", "x", "y", "bbox_width", "bbox_height"],
    )
    result["capture_group"] = result.file_name.str.split("_").str[0]
    result["relative_area"] = (
        result.bbox_width * result.bbox_height / (result.width * result.height)
    )
    result["aspect_ratio"] = result.bbox_width / result.bbox_height
    result["center_x"] = (result.x + result.bbox_width / 2) / result.width
    result["center_y"] = (result.y + result.bbox_height / 2) / result.height
    return result, lookup


def draw_boxes(records: pd.DataFrame, names: list[str], destination: Path) -> None:
    """이미지별로 연결된 유효 bbox와 제품명을 그린다."""
    figure, axes = plt.subplots(
        1, max(1, len(names)), figsize=(7 * max(1, len(names)), 7), squeeze=False
    )
    for axis in axes.flat:
        axis.axis("off")
    for axis, name in zip(axes.flat, names, strict=False):
        subset = records.loc[
            (records.file_name == name) & records.image_present & ~records.duplicate
        ]
        if subset.empty:
            continue
        img = cv2.imread(str(subset.iloc[0].image_path))
        if img is None:
            axis.set_title(f"Decode failed: {name}")
            continue
        axis.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axis.set_title(name, fontsize=8)
        for row in subset.loc[subset.bbox_valid].itertuples():
            axis.add_patch(
                Rectangle(
                    (row.x, row.y), row.bbox_width, row.bbox_height, fill=False, edgecolor="red"
                )
            )
            axis.text(row.x, row.y, row.category_name, color="red", fontsize=8)
    figure.tight_layout()
    figure.savefig(destination, dpi=140)
    plt.close(figure)


def inspect_images(paths: list[Path]) -> pd.DataFrame:
    """이미지를 한 장씩 읽어 체크섬과 해상도 및 품질 지표를 계산한다."""
    columns = [
        "file_name",
        "path",
        "readable",
        "sha256",
        "actual_width",
        "actual_height",
        "blur_score",
        "brightness",
        "average_hash",
        "error",
    ]
    rows = []
    for path in paths:
        row = dict.fromkeys(columns, np.nan)
        row.update(
            file_name=path.name,
            path=str(path),
            readable=False,
            sha256=None,
            average_hash=None,
            error="",
        )
        try:
            with path.open("rb") as stream:
                row["sha256"] = hashlib.file_digest(stream, "sha256").hexdigest()
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise ValueError("이미지 디코딩 실패")
            small = cv2.resize(img, (8, 8), interpolation=cv2.INTER_AREA)
            average_hash = sum(
                int(bit) << index for index, bit in enumerate((small > small.mean()).flat)
            )
            row.update(
                readable=True,
                actual_width=img.shape[1],
                actual_height=img.shape[0],
                blur_score=float(cv2.Laplacian(img, cv2.CV_64F).var()),
                brightness=float(img.mean()),
                average_hash=average_hash,
            )
        except (OSError, ValueError, cv2.error) as error:
            row["error"] = str(error)
        rows.append(row)
    result = pd.DataFrame(rows, columns=columns)
    result["readable"] = result.readable.astype(bool)
    # 64비트 해시가 결측값 때문에 실수로 변환되어 정밀도를 잃지 않도록 한다.
    result["average_hash"] = pd.Series([row["average_hash"] for row in rows], dtype=object)
    return result


def plot_distributions(objects: pd.DataFrame, quality: pd.DataFrame, destination: Path) -> None:
    """객체 수와 유효 bbox 및 이미지 밝기 분포를 저장한다."""
    figure, axes = plt.subplots(2, 3, figsize=(16, 9))
    values = [
        (objects.groupby("category_id").size(), "Objects per class"),
        (objects.groupby("file_name").size(), "Objects per image"),
        (objects.relative_area, "Relative bbox area"),
        (objects.aspect_ratio, "Bbox aspect ratio"),
        (objects.center_x, "Normalized bbox center X"),
        (quality.loc[quality.readable, "brightness"], "Image brightness"),
    ]
    for axis, (series, title) in zip(axes.flat, values, strict=True):
        axis.hist(series.replace([np.inf, -np.inf], np.nan).dropna(), bins=30)
        axis.set_title(title)
    figure.tight_layout()
    figure.savefig(destination, dpi=140)
    plt.close(figure)


def near_duplicate_candidates(quality: pd.DataFrame, max_distance: int = 3) -> pd.DataFrame:
    """해시 분할 색인으로 해밍 거리가 작은 이미지 쌍을 찾는다."""
    if not 0 <= max_distance < 64:
        raise ValueError("해시 거리는 0 이상 64 미만이어야 합니다.")
    rows = list(quality.loc[quality.readable].itertuples())
    buckets, pairs = {}, []
    # 거리 이하의 비트만 다르면 거리+1개 구간 중 적어도 하나는 일치한다.
    boundaries = np.linspace(0, 64, max_distance + 2, dtype=int)
    for index, row in enumerate(rows):
        value = int(row.average_hash)
        candidates, keys = set(), []
        for part, (start, stop) in enumerate(zip(boundaries[:-1], boundaries[1:], strict=True)):
            key = (part, (value >> int(start)) & ((1 << int(stop - start)) - 1))
            candidates.update(buckets.get(key, []))
            keys.append(key)
        for other in sorted(candidates):
            previous = rows[other]
            distance = (value ^ int(previous.average_hash)).bit_count()
            if distance <= max_distance:
                pairs.append({"image_a": previous.path, "image_b": row.path, "distance": distance})
        for key in keys:
            buckets.setdefault(key, []).append(index)
    return pd.DataFrame(pairs, columns=["image_a", "image_b", "distance"])


def suggest_split(
    objects: pd.DataFrame, seed: int = 42, validation_fraction: float = 0.2
) -> tuple[pd.DataFrame, pd.Series]:
    """촬영 그룹을 유지한 재현 가능한 검증 후보와 클래스 커버리지를 반환한다."""
    if not 0 < validation_fraction < 1:
        raise ValueError("검증 비율은 0과 1 사이여야 합니다.")
    groups = sorted(objects.capture_group.unique())
    if len(groups) < 2:
        validation_groups = set()
    else:
        count = min(len(groups) - 1, max(1, round(len(groups) * validation_fraction)))
        validation_groups = set(np.random.default_rng(seed).choice(groups, count, replace=False))
    result = objects[["file_name", "capture_group"]].drop_duplicates().copy()
    result["split"] = np.where(result.capture_group.isin(validation_groups), "validation", "train")
    validation = objects.capture_group.isin(validation_groups)
    train_classes = set(objects.loc[~validation, "category_id"])
    validation_classes = set(objects.loc[validation, "category_id"])
    info = pd.Series(
        {
            "seed": seed,
            "groups": len(groups),
            "validation_groups": len(validation_groups),
            "train_images": int((result.split == "train").sum()),
            "validation_images": int((result.split == "validation").sum()),
            "validation_only_classes": sorted(validation_classes - train_classes, key=str),
            "train_only_classes": sorted(train_classes - validation_classes, key=str),
            "status": "검수 전 후보" if len(groups) >= 2 else "그룹 부족: 분할 불가",
        }
    )
    return result, info


def write_report(
    records: pd.DataFrame,
    json_files: list[Path],
    load_issues: pd.DataFrame,
    train_quality: pd.DataFrame,
    test_quality: pd.DataFrame,
    objects: pd.DataFrame,
    split_info: pd.Series,
    near_pairs: pd.DataFrame,
    destination: Path,
) -> None:
    """무결성 수치와 분할 후보를 마크다운 보고서에 기록한다."""
    train_hashes = set(train_quality.sha256.dropna())
    test_hashes = set(test_quality.sha256.dropna())
    summary = {
        "JSON 파일": len(json_files),
        "읽기 및 참조 오류": len(load_issues),
        "전체 객체": len(records),
        "이미지 연결 객체(중복 제외)": len(objects),
        "고유 클래스": objects.category_id.nunique(),
        "중복 객체": int(records.duplicate.sum()),
        "잘못된 bbox": int((~records.bbox_valid).sum()),
        "면적 불일치": int((~records.area_matches).sum()),
        "동명 이미지로 연결 불가한 객체": int(records.ambiguous_image.sum()),
        "학습 이미지": len(train_quality),
        "테스트 이미지": len(test_quality),
        "학습 디코딩 실패": int((~train_quality.readable).sum()),
        "테스트 디코딩 실패": int((~test_quality.readable).sum()),
        "학습/테스트 공통 SHA256": len(train_hashes & test_hashes),
        "학습 내 동일 파일 추가 사본": int(train_quality.sha256.dropna().duplicated().sum()),
        "유사 이미지 쌍 후보": len(near_pairs),
    }
    lines = ["# 소희 EDA 결과", "", "## 규모와 무결성", ""]
    lines.extend(f"- {key}: {value}" for key, value in summary.items())
    lines.extend(["", "## 분할 후보", ""])
    lines.extend(f"- {key}: {value}" for key, value in split_info.items())
    lines.extend(
        [
            "",
            "## 검수 사항",
            "",
            "- bbox는 xywh로 해석했습니다. 시각 검수가 필요합니다.",
            "- 촬영 그룹은 파일명 첫 밑줄 앞 값이며 정식 세션 ID가 아닙니다.",
            "- 객체가 없는 이미지의 누락 라벨 여부와 유사 이미지 후보는 사람이 확인합니다.",
            "- 흐림 지표는 배경 영향을 포함하며 테스트 클래스는 추정하지 않았습니다.",
            "- 분할은 후보이며 동일/유사 이미지의 그룹 통합 및 클래스 커버리지 검수가 필요합니다.",
        ]
    )
    if not load_issues.empty:
        lines.extend(
            [
                "",
                "## 읽기 및 참조 오류 (최대 20건)",
                "",
                "```text",
                load_issues.head(20).to_string(index=False),
                "```",
            ]
        )
    destination.write_text("\n".join(lines) + "\n", encoding="utf-8")

In [32]:
SEED = 42
output_dir = PROJECT_ROOT / "notebooks" / "sohee_eda_figures"
output_dir.mkdir(exist_ok=True)
train_paths = image_paths(PROJECT_ROOT / "data" / "raw/train_images")
test_paths = image_paths(PROJECT_ROOT / "data" / "raw/test_images")
print({"train_images": len(train_paths), "test_images": len(test_paths), "seed": SEED})

{'train_images': 232, 'test_images': 842, 'seed': 42}


## 1. 샘플 구조 확인
처음 세 상위 폴더에서 JSON 하나씩 읽어 자료형·이미지 ID·클래스·bbox를 확인합니다.
전역 image_id로 결합하지 않습니다. 전체 분석에서도 JSON 내부 참조를 먼저 확인합니다.


In [33]:
annotation_dir = PROJECT_ROOT / "data" / "raw/train_annotations"
sample_rows = []
for folder in sorted(annotation_dir.iterdir())[:3]:
    if not folder.is_dir():
        continue
    sample = next(iter(sorted(folder.rglob("*.json"))), None)
    if sample is None:
        continue
    payload = json.loads(sample.read_text(encoding="utf-8-sig"))
    sample_rows.append(
        {
            "source": str(sample.relative_to(PROJECT_ROOT)),
            "types": {
                key: type(payload[key]).__name__ for key in ["images", "annotations", "categories"]
            },
            "images": payload["images"],
            "annotations": payload["annotations"],
            "categories": payload["categories"],
        }
    )
display(pd.DataFrame(sample_rows))

,source,types,images,annotations,categories
0,data/raw/train_annotations/K-001900-016548-019...,"{'images': 'list', 'annotations': 'list', 'cat...",[{'file_name': 'K-001900-016548-019607-029451_...,"[{'area': 35910, 'iscrowd': 0, 'bbox': [644, 8...","[{'supercategory': 'pill', 'id': 1900, 'name':..."
1,data/raw/train_annotations/K-001900-016548-019...,"{'images': 'list', 'annotations': 'list', 'cat...",[{'file_name': 'K-001900-016548-019607-033009_...,"[{'area': 34736, 'iscrowd': 0, 'bbox': [642, 8...","[{'supercategory': 'pill', 'id': 1900, 'name':..."
2,data/raw/train_annotations/K-001900-016548-021...,"{'images': 'list', 'annotations': 'list', 'cat...",[{'file_name': 'K-001900-016548-021771-027926_...,"[{'area': 28968, 'iscrowd': 0, 'bbox': [629, 8...","[{'supercategory': 'pill', 'id': 1900, 'name':..."


## 2. 이미지 단위 연결 기준 확정
전체 JSON 메타데이터를 순차적으로 읽습니다. 파일명을 고유 이미지와 대응시키고 출처를 유지합니다.
동일 이미지·클래스·bbox 중복은 표시만 하며, 분포용 테이블에서 제외합니다.
객체가 없는 이미지가 실제 음성 이미지인지 라벨 누락인지는 자동 판단하지 않습니다.


In [34]:
records, json_files, load_issues = load_annotations(annotation_dir)
records, image_lookup = prepare_tables(records, train_paths)
objects = records.loc[records.image_present & ~records.duplicate].copy()
if objects.empty:
    raise RuntimeError("이미지와 연결된 객체가 없습니다. 경로와 파일명을 확인하세요.")
print({"json": len(json_files), "records": len(records), "linked_objects": len(objects)})
display(records.groupby("image_id").file_name.nunique().sort_values(ascending=False).head())
display(
    records.groupby("file_name")
    .agg(
        json_count=("source", "nunique"),
        image_ids=("image_id", "nunique"),
        classes=("category_id", "nunique"),
    )
    .head()
)
display(load_issues.head(10))

{'json': 763, 'records': 763, 'linked_objects': 763}


image_id
14    1
16    1
18    1
34    1
35    1
Name: file_name, dtype: int64

,json_count,image_ids,classes
file_name,,,
K-001900-016548-019607-029451_0_2_0_2_70_000_200.png,4,1,4
K-001900-016548-019607-029451_0_2_0_2_75_000_200.png,4,1,4
K-001900-016548-019607-029451_0_2_0_2_90_000_200.png,4,1,4
K-001900-016548-019607-033009_0_2_0_2_70_000_200.png,4,1,4
K-001900-016548-019607-033009_0_2_0_2_75_000_200.png,4,1,4


,source,issue


## 3. bbox 시각 검증
서로 다른 알약 조합의 이미지 최대 세 장을 선택하고 모든 연결된 bbox와 제품명을 표시합니다.
xywh 해석, 이미지 회전, 박스 위치와 제품명을 사람이 확인해야 합니다.


In [ ]:
sample_names = (
    objects[["file_name", "capture_group"]]
    .drop_duplicates()
    .drop_duplicates("capture_group")
    .head(3)
    .file_name.tolist()
)
draw_boxes(records, sample_names, output_dir / "bbox_samples.png")
display(Image(filename=str(output_dir / "bbox_samples.png")))

/tmp/ipykernel_35880/4041253958.py:179: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_35880/4041253958.py:179: UserWarning: Glyph 47161 (\N{HANGUL SYLLABLE RYEONG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_35880/4041253958.py:179: UserWarning: Glyph 48512 (\N{HANGUL SYLLABLE BU}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_35880/4041253958.py:179: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_35880/4041253958.py:179: UserWarning: Glyph 54028 (\N{HANGUL SYLLABLE PA}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_35880/4041253958.py:179: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_35880/4041253958.py:179: UserWarning: Glyph 44032 (\N{HANGUL SYLLABLE GA}) missing from f

## 4. 기본 규모와 무결성 분석
모든 학습·테스트 이미지를 한 장씩 디코딩하고 SHA256·해상도·품질 지표를 계산합니다.
배경을 포함한 Laplacian 분산은 흐림 확정값이 아닙니다. 테스트 클래스는 추정하지 않습니다.


In [ ]:
train_quality = inspect_images(train_paths)
test_quality = inspect_images(test_paths)
annotated_names = set(records.file_name)
actual_names = set(train_quality.file_name)
display(
    pd.Series(
        {
            "annotation_images": len(annotated_names),
            "annotation_without_image": len(annotated_names - actual_names),
            "image_without_annotation": len(actual_names - annotated_names),
            "invalid_bbox": int((~records.bbox_valid).sum()),
            "area_mismatch": int((~records.area_matches).sum()),
            "duplicate_records": int(records.duplicate.sum()),
            "train_decode_failures": int((~train_quality.readable).sum()),
            "test_decode_failures": int((~test_quality.readable).sum()),
        }
    )
)
display(
    records.loc[
        ~records.bbox_valid | ~records.area_matches,
        ["source", "file_name", "category_id", "bbox_valid", "area_matches"],
    ].head(20)
)
display(train_quality.groupby(["actual_width", "actual_height"]).size())

## 5. 클래스·객체·bbox 분포 분석
전체 annotation과 로컬 이미지 연결 객체를 구분합니다. 아래 그래프는 연결된 객체 기준입니다.
클래스별 이미지 수와 객체 수를 별도로 계산하고, 좌표 통계에는 유효한 bbox만 사용합니다.


In [ ]:
class_counts = (
    objects.groupby(["category_id", "category_name"])
    .agg(
        objects=("file_name", "size"),
        images=("file_name", "nunique"),
        groups=("capture_group", "nunique"),
    )
    .sort_values("objects", ascending=False)
)
display(class_counts)
display(records.groupby("category_id").size().describe().rename("전체 annotation 클래스당 객체 수"))
display(objects.groupby("file_name").size().describe().rename("연결된 이미지당 객체 수"))
plot_distributions(objects.loc[objects.bbox_valid], train_quality, output_dir / "distributions.png")
display(Image(filename=str(output_dir / "distributions.png")))

## 6. 촬영 조건과 어려운 사례
촬영 조건별 이미지 수와 클래스 교차표를 확인합니다. 앞뒷면은 제품별 값일 수 있습니다.
흐림 후보를 시각 검토하고, 평균 해시 거리 3 이하의 쌍을 유사 이미지 후보로만 취급합니다.


In [ ]:
condition_tables = {}
for condition in CONDITIONS:
    values = objects[["file_name", condition]].drop_duplicates()
    print(condition)
    display(values[condition].fillna("<결측>").replace("", "<빈 문자열>").value_counts())
    condition_tables[condition] = pd.crosstab(objects.category_id, objects[condition])
    display(condition_tables[condition].head(10))
near_pairs = near_duplicate_candidates(train_quality)
print("평균 해시 유사 쌍 후보:", len(near_pairs))
display(near_pairs.head(10))
readable_names = set(train_quality.loc[train_quality.readable, "file_name"])
linked_quality = train_quality.loc[
    train_quality.file_name.isin(set(objects.file_name) & readable_names)
]
difficult_names = linked_quality.nsmallest(3, "blur_score").file_name.tolist()
draw_boxes(records, difficult_names, output_dir / "difficult_samples.png")
display(Image(filename=str(output_dir / "difficult_samples.png")))

## 7. 분할 기준과 학습 준비 사항
파일명 첫 밑줄 앞의 알약 조합을 촬영 그룹의 대리값으로 사용합니다.
seed=42로 그룹의 약 20%를 검증 후보로 선택하고 학습에 없는 검증 클래스를 확인합니다.
정식 세션 ID·동일/유사 이미지 검수 및 클래스 커버리지 조정 전에는 학습에 사용하지 않습니다.
파일 이동·학습·라벨 변환은 하지 않습니다. 원본이 다른 이미지라도 중복이면 그룹을 합쳐야 합니다.


In [ ]:
split_candidates, split_info = suggest_split(objects, seed=SEED)
display(pd.crosstab(split_candidates.capture_group, split_candidates.split))
display(split_info)
report_path = PROJECT_ROOT / "notebooks" / "sohee_eda_results.md"
write_report(
    records,
    json_files,
    load_issues,
    train_quality,
    test_quality,
    objects,
    split_info,
    near_pairs,
    report_path,
)
print("결과 보고서:", report_path.relative_to(PROJECT_ROOT))

## 검수와 다음 작업
- bbox 시각 검증 후 좌표 해석을 확정합니다.
- 누락 이미지·누락 라벨·bbox 오류·클래스 이름 충돌은 자동 수정하지 않습니다.
- 흐림·반사·가림·유사 제품은 후보 이미지를 사람이 검토합니다.
- 희소 클래스는 이미지 수뿐 아니라 독립 촬영 그룹 수로 평가합니다.
- 색상·각인을 보존하는 범위에서 밝기·대비·회전 등의 증강 후보를 검토합니다.
- 노트북 출력은 지운 상태로 유지합니다. 보고서와 그림은 셀을 실행하면 갱신됩니다.
